# r-GCN Dempster-Shafer ESM Identification Pipeline

This notebook trains an r-GCN over the existing aircraft/radar knowledge graph plus synthetic ESM observation nodes from `esm_observations_notebook_demo.json`. Observation node features are limited to measured ESM and kinematic fields. Ground-truth aircraft variant and radar mode labels are used only to build training targets and score validation/test splits.

In [ ]:
from pathlib import Path
import json, math, random
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    SummaryWriter = None
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, desc=None):
        total = len(iterable) if hasattr(iterable, "__len__") else None
        label = desc or "Progress"
        print(f"{label}: starting" + (f" ({total} steps)" if total else ""))
        for item in iterable:
            yield item
        print(f"{label}: done")
from rgcn_fusion.model import RGCNLayer
from rgcn_fusion.dempster_shafer import belief_plausibility, validate_masses
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


## User-controlled configuration

In [ ]:
OBS_PATH = Path("C:/Users/theon/r-GCN_fusion/notebooks/generated/esm_observations_notebook_demo.json")
KG_PATH = Path("C:\\Users\\theon\\r-GCN_fusion\\generated\\aircraft_radar_kg.json")
ARTIFACT_DIR = Path("artifacts/notebook_rgcn_ds")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
TENSORBOARD_LOG_DIR = ARTIFACT_DIR / "tensorboard"
TRAIN_FRACTION = 0.5
TEST_FRACTION = 0.3
VAL_FRACTION = 0.2
EPOCHS = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_FEATURES = 64
DROPOUT = 0.15
PATIENCE = 30
UNCERTAINTY_MASS = 0.15
MAX_FULL_DS_AIRCRAFT_CLASSES = 10
assert abs(TRAIN_FRACTION + TEST_FRACTION + VAL_FRACTION - 1.0) < 1e-9


## TensorBoard tracking

Training writes TensorBoard event files under `artifacts/notebook_rgcn_ds/tensorboard`. In Jupyter, run the cell below before or after training to open the TensorBoard UI. Scalars include total train loss, validation loss, validation aircraft accuracy, validation mode accuracy, and learning rate.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {TENSORBOARD_LOG_DIR}


## Load existing KG and synthetic observations

In [ ]:
with OBS_PATH.open("r", encoding="utf-8") as f:
    obs_payload = json.load(f)
observations = obs_payload["observations"]
with KG_PATH.open("r", encoding="utf-8") as f:
    kg = json.load(f)
print(f"Loaded {len(observations)} observations, {len(kg['nodes'])} KG nodes, {len(kg['edges'])} KG edges")


## Build measured-only observation features

The model sees only measured ESM radar parameters, observed waveform/scan type, approximate kinematics, timestamp, and estimated emitter location. It never receives `ground_truth_label` values, aircraft IDs, radar IDs, or candidate labels as input features.

In [ ]:
NUMERIC_FEATURES = [
    ("esm_radar_parameters", "measured_centre_frequency_ghz", "value"), ("esm_radar_parameters", "measured_centre_frequency_ghz", "error"),
    ("esm_radar_parameters", "measured_bandwidth_mhz", "value"), ("esm_radar_parameters", "measured_bandwidth_mhz", "error"),
    ("esm_radar_parameters", "measured_prf_hz", "value"), ("esm_radar_parameters", "measured_prf_hz", "error"),
    ("esm_radar_parameters", "measured_pulse_width_us", "value"), ("esm_radar_parameters", "measured_pulse_width_us", "error"),
    ("esm_radar_parameters", "measured_pulse_repetition_interval_us", "value"), ("esm_radar_parameters", "measured_pulse_repetition_interval_us", "error"),
    ("esm_radar_parameters", "measured_duty_cycle", "value"), ("esm_radar_parameters", "measured_duty_cycle", "error"),
    ("esm_radar_parameters", "measured_dwell_time_ms", "value"), ("esm_radar_parameters", "measured_dwell_time_ms", "error"),
    ("esm_radar_parameters", "measured_coherent_processing_interval_ms", "value"), ("esm_radar_parameters", "measured_coherent_processing_interval_ms", "error"),
    ("approximate_kinematics", "altitude_m"), ("approximate_kinematics", "altitude_error_m"),
    ("approximate_kinematics", "ground_speed_kph"), ("approximate_kinematics", "ground_speed_error_kph"),
    ("approximate_kinematics", "heading_deg"), ("approximate_kinematics", "heading_error_deg"),
    ("estimated_emitter_location", "estimated_latitude_deg"), ("estimated_emitter_location", "estimated_longitude_deg"),
]
def get_nested(d, path):
    cur = d
    for key in path:
        cur = cur.get(key, {}) if isinstance(cur, dict) else {}
    return float(cur) if cur != {} else 0.0
scan_values = sorted({o["esm_radar_parameters"].get("observed_scan_type", "unknown") for o in observations})
wave_values = sorted({o["esm_radar_parameters"].get("observed_waveform", "unknown") for o in observations})
def measured_feature_vector(o):
    nums = [get_nested(o, p) for p in NUMERIC_FEATURES]
    scan = o["esm_radar_parameters"].get("observed_scan_type", "unknown")
    wave = o["esm_radar_parameters"].get("observed_waveform", "unknown")
    cats = [1.0 if scan == v else 0.0 for v in scan_values] + [1.0 if wave == v else 0.0 for v in wave_values]
    return nums + cats
obs_features = np.array([measured_feature_vector(o) for o in observations], dtype=np.float32)
mu, sigma = obs_features.mean(axis=0, keepdims=True), obs_features.std(axis=0, keepdims=True)
sigma[sigma == 0] = 1.0
obs_features = (obs_features - mu) / sigma
obs_features.shape


## Construct a heterogeneous graph for r-GCN

KG edges are retained. Observation nodes are connected to radar-mode nodes whose public mode envelopes are compatible with measured ESM values. These compatibility edges are derived from measured fields only.

In [ ]:
node_ids = [n["id"] for n in kg["nodes"]] + [o["observation_id"] for o in observations]
node_index = {nid: i for i, nid in enumerate(node_ids)}
feature_dim = obs_features.shape[1]
features = np.zeros((len(node_ids), feature_dim), dtype=np.float32)
features[len(kg["nodes"]):] = obs_features
relations = sorted({e["relation"] for e in kg["edges"]}) + ["OBS_COMPATIBLE_WITH_MODE", "REV_OBS_COMPATIBLE_WITH_MODE"]
rel_index = {r: i for i, r in enumerate(relations)}
edge_src, edge_dst, edge_type = [], [], []
def add_edge(s, t, r):
    if s in node_index and t in node_index:
        edge_src.append(node_index[s]); edge_dst.append(node_index[t]); edge_type.append(rel_index[r])
for e in kg["edges"]:
    add_edge(e["source"], e["target"], e["relation"])
def mode_compatible(obs, mode_node):
    p = mode_node.get("properties", {})
    esm = obs["esm_radar_parameters"]
    checks = [("measured_centre_frequency_ghz", "centre_frequency_min_ghz", "centre_frequency_max_ghz"), ("measured_bandwidth_mhz", "bandwidth_min_mhz", "bandwidth_max_mhz"), ("measured_prf_hz", "prf_min_hz", "prf_max_hz"), ("measured_pulse_width_us", "pulse_width_min_us", "pulse_width_max_us"), ("measured_duty_cycle", "duty_cycle_min", "duty_cycle_max"), ("measured_dwell_time_ms", "dwell_time_min_ms", "dwell_time_max_ms"), ("measured_coherent_processing_interval_ms", "coherent_processing_interval_min_ms", "coherent_processing_interval_max_ms")]
    score = 0
    for measured_key, lo_key, hi_key in checks:
        if measured_key not in esm or lo_key not in p or hi_key not in p:
            continue
        lo, hi = esm[measured_key].get("min", esm[measured_key]["value"]), esm[measured_key].get("max", esm[measured_key]["value"])
        if max(lo, p[lo_key]) <= min(hi, p[hi_key]): score += 1
    score += int(p.get("scan_type") == esm.get("observed_scan_type"))
    score += int(p.get("waveform") == esm.get("observed_waveform"))
    return score >= 3
mode_nodes = [n for n in kg["nodes"] if n.get("label") == "RadarMode"]
for o in observations:
    matches = [m for m in mode_nodes if mode_compatible(o, m)] or mode_nodes
    for m in matches:
        add_edge(o["observation_id"], m["id"], "OBS_COMPATIBLE_WITH_MODE")
        add_edge(m["id"], o["observation_id"], "REV_OBS_COMPATIBLE_WITH_MODE")
edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long, device=DEVICE)
edge_type_t = torch.tensor(edge_type, dtype=torch.long, device=DEVICE)
x = torch.tensor(features, dtype=torch.float32, device=DEVICE)
print(x.shape, edge_index.shape, len(relations))


## Frequency-gated Dempster-Shafer frames

Before constructing any Dempster-Shafer targets, each observation's measured radar centre frequency is compared with the `centre_frequency_min_ghz` / `centre_frequency_max_ghz` ranges on KG `RadarMode` nodes. Matching radar modes identify compatible radar variants through `HAS_MODE`, and compatible aircraft variants through `USES_RADAR`. The notebook uses the union of these KG-compatible candidates as the scoreable frame. If no KG mode matches an observation frequency, the notebook falls back to the full KG frame for that observation and records the fallback in `frequency_gate_summary`.

When the frequency-gated aircraft frame has at most `MAX_FULL_DS_AIRCRAFT_CLASSES` variants, mass vectors keep the repository's full Dempster-Shafer subset convention: every non-empty subset of `n` hypotheses is represented by bit masks `1..2**n - 1`. If the aircraft frame remains larger than that threshold, this notebook declares the frame excessive and switches both tasks to scalable singleton-plus-uncertainty targets of length `n_classes + 1`; the final element represents residual uncertainty over the frame.

For a frame with `n` singleton hypotheses, mass vectors follow the repository convention implemented by `rgcn_fusion.dempster_shafer`: every non-empty subset of the frame is represented by a bit mask from `1` through `2**n - 1`. Singleton class `i` is stored at index `(1 << i) - 1`, and the final element represents the full frame (total uncertainty). During target construction, each labeled observation assigns `1 - UNCERTAINTY_MASS` to the true singleton and `UNCERTAINTY_MASS` to the full frame. At inference time, the model predicts a normalized mass vector over the same subset ordering, and `belief_plausibility(...)` converts those masses into singleton Belief-Plausibility intervals for reporting.

Ground-truth aircraft variants and modes are therefore used only to define scoreable output frames and supervised targets. They are not included in the measured-only feature vectors or observation-to-KG compatibility edges used as model inputs.

## Labels, split, and Dempster-Shafer target masses

In [ ]:
node_by_id = {n["id"]: n for n in kg["nodes"]}
radar_modes_by_id = {n["id"]: n for n in kg["nodes"] if n.get("label") == "RadarMode"}
radars_by_id = {n["id"]: n for n in kg["nodes"] if n.get("label") == "Radar"}
aircraft_by_id = {n["id"]: n for n in kg["nodes"] if n.get("label") == "AircraftVariant"}
radar_to_modes = {}
mode_to_radar = {}
radar_to_aircraft = {}
for edge in kg["edges"]:
    if edge["relation"] == "HAS_MODE":
        radar_to_modes.setdefault(edge["source"], set()).add(edge["target"])
        mode_to_radar[edge["target"]] = edge["source"]
    elif edge["relation"] == "USES_RADAR":
        radar_to_aircraft.setdefault(edge["target"], set()).add(edge["source"])
all_kg_mode_ids = set(radar_modes_by_id)
all_kg_radar_ids = set(radars_by_id)
all_kg_aircraft_ids = set(aircraft_by_id)

def frequency_value_ghz(obs):
    return obs["esm_radar_parameters"]["measured_centre_frequency_ghz"]["value"]

def mode_matches_frequency(mode_node, frequency_ghz):
    props = mode_node["properties"]
    return props["centre_frequency_min_ghz"] <= frequency_ghz <= props["centre_frequency_max_ghz"]

def frequency_candidate_ids(obs):
    frequency_ghz = frequency_value_ghz(obs)
    mode_ids = {mode_id for mode_id, mode_node in radar_modes_by_id.items() if mode_matches_frequency(mode_node, frequency_ghz)}
    used_fallback = False
    if not mode_ids:
        mode_ids = set(all_kg_mode_ids)
        used_fallback = True
    radar_ids = {mode_to_radar[mode_id] for mode_id in mode_ids if mode_id in mode_to_radar}
    aircraft_ids = {aircraft_id for radar_id in radar_ids for aircraft_id in radar_to_aircraft.get(radar_id, set())}
    if not aircraft_ids:
        radar_ids = set(all_kg_radar_ids)
        aircraft_ids = set(all_kg_aircraft_ids)
        used_fallback = True
    return {"frequency_ghz": frequency_ghz, "mode_ids": mode_ids, "radar_ids": radar_ids, "aircraft_ids": aircraft_ids, "used_fallback": used_fallback}

frequency_candidates = [frequency_candidate_ids(o) for o in observations]
frequency_gated_aircraft_ids = set().union(*(c["aircraft_ids"] for c in frequency_candidates))
frequency_gated_mode_ids = set().union(*(c["mode_ids"] for c in frequency_candidates))
frequency_gated_radar_ids = set().union(*(c["radar_ids"] for c in frequency_candidates))
# Ensure supervised truth labels remain scoreable if measurement noise or KG coverage excludes them.
truth_aircraft_variants = {o["ground_truth_label"]["aircraft_variant"] for o in observations}
truth_modes = {o["ground_truth_label"]["mode"] for o in observations}
aircraft_classes = sorted({aircraft_by_id[aircraft_id]["properties"]["variant"] for aircraft_id in frequency_gated_aircraft_ids if aircraft_id in aircraft_by_id} | truth_aircraft_variants)
mode_classes = sorted({radar_modes_by_id[mode_id]["properties"]["name"] for mode_id in frequency_gated_mode_ids if mode_id in radar_modes_by_id} | truth_modes)
aircraft_to_idx = {c: i for i, c in enumerate(aircraft_classes)}
mode_to_idx = {c: i for i, c in enumerate(mode_classes)}
use_singleton_uncertainty_masses = len(aircraft_classes) > MAX_FULL_DS_AIRCRAFT_CLASSES
if use_singleton_uncertainty_masses:
    print(f"Frequency-gated aircraft frame has {len(aircraft_classes)} variants (> {MAX_FULL_DS_AIRCRAFT_CLASSES}); declaring this notebook frame excessive and using n_classes + 1 singleton-plus-uncertainty masses.")
else:
    print(f"Frequency-gated aircraft frame has {len(aircraft_classes)} variants; using full Dempster-Shafer subset masses.")
frequency_gate_summary = {"candidate_aircraft_variants": len(aircraft_classes), "candidate_radar_variants": len(frequency_gated_radar_ids), "candidate_radar_modes": len(frequency_gated_mode_ids), "observations_using_frequency_fallback": sum(c["used_fallback"] for c in frequency_candidates), "mass_representation": "singleton_plus_uncertainty" if use_singleton_uncertainty_masses else "full_powerset"}
frequency_gate_summary

def full_powerset_singleton_mass(class_idx, n_classes, uncertainty=UNCERTAINTY_MASS):
    masses = np.zeros(2**n_classes - 1, dtype=np.float32)
    masses[(1 << class_idx) - 1] = 1.0 - uncertainty
    masses[-1] = uncertainty
    return masses

def singleton_plus_uncertainty_mass(class_idx, n_classes, uncertainty=UNCERTAINTY_MASS):
    masses = np.zeros(n_classes + 1, dtype=np.float32)
    masses[class_idx] = 1.0 - uncertainty
    masses[-1] = uncertainty
    return masses

def target_mass(class_idx, n_classes):
    if use_singleton_uncertainty_masses:
        return singleton_plus_uncertainty_mass(class_idx, n_classes)
    return full_powerset_singleton_mass(class_idx, n_classes)

def singleton_score_indices(n_classes):
    if use_singleton_uncertainty_masses:
        return list(range(n_classes))
    return [(1 << i) - 1 for i in range(n_classes)]

y_air = np.stack([target_mass(aircraft_to_idx[o["ground_truth_label"]["aircraft_variant"]], len(aircraft_classes)) for o in observations])
y_mode = np.stack([target_mass(mode_to_idx[o["ground_truth_label"]["mode"]], len(mode_classes)) for o in observations])
obs_node_indices = np.arange(len(kg["nodes"]), len(node_ids))
perm = np.random.default_rng(SEED).permutation(len(observations))
n_train = int(round(TRAIN_FRACTION * len(perm))); n_test = int(round(TEST_FRACTION * len(perm)))
splits = {"train": torch.tensor(obs_node_indices[perm[:n_train]], dtype=torch.long, device=DEVICE), "test": torch.tensor(obs_node_indices[perm[n_train:n_train+n_test]], dtype=torch.long, device=DEVICE), "val": torch.tensor(obs_node_indices[perm[n_train+n_test:]], dtype=torch.long, device=DEVICE)}
obs_pos = {node_idx: i for i, node_idx in enumerate(obs_node_indices)}
split_obs_rows = {k: torch.tensor([obs_pos[int(i.cpu())] for i in v], dtype=torch.long, device=DEVICE) for k, v in splits.items()}
print({k: len(v) for k, v in splits.items()}, len(aircraft_classes), len(mode_classes), frequency_gate_summary)


In [ ]:
aircraft_classes = sorted({o["ground_truth_label"]["aircraft_variant"] for o in observations})
mode_classes = sorted({o["ground_truth_label"]["mode"] for o in observations})
aircraft_to_idx = {c: i for i, c in enumerate(aircraft_classes)}
mode_to_idx = {c: i for i, c in enumerate(mode_classes)}

print("observations:", len(observations))
print("aircraft classes:", len(aircraft_classes))
print("mode classes:", len(mode_classes))

print("aircraft mass length:", 2**len(aircraft_classes) - 1)
print("mode mass length:", 2**len(mode_classes) - 1)

print("first few aircraft classes:")
print(aircraft_classes[:20])

## Multi-head r-GCN model

In [ ]:
class MultiTaskDSEvidenceModel(nn.Module):
    def __init__(self, in_features, hidden_features, num_relations, n_aircraft, n_modes, dropout=0.1):
        super().__init__()
        self.conv1 = RGCNLayer(in_features, hidden_features, max(num_relations, 1))
        self.conv2 = RGCNLayer(hidden_features, hidden_features, max(num_relations, 1))
        self.dropout = nn.Dropout(dropout)
        self.aircraft_head = nn.Linear(hidden_features, y_air.shape[1])
        self.mode_head = nn.Linear(hidden_features, y_mode.shape[1])
    def forward(self, x, edge_index, edge_type):
        h = F.relu(self.conv1(x, edge_index, edge_type)); h = self.dropout(h); h = F.relu(self.conv2(h, edge_index, edge_type))
        return F.softmax(self.aircraft_head(h), dim=-1), F.softmax(self.mode_head(h), dim=-1)
model = MultiTaskDSEvidenceModel(feature_dim, HIDDEN_FEATURES, len(relations), len(aircraft_classes), len(mode_classes), DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
y_air_t = torch.tensor(y_air, dtype=torch.float32, device=DEVICE); y_mode_t = torch.tensor(y_mode, dtype=torch.float32, device=DEVICE)


## Train with progress bar, validation, and improving checkpoints

In [ ]:
def ds_kl(pred, target): return F.kl_div(torch.log(pred.clamp_min(1e-9)), target, reduction="batchmean")
def eval_split(split):
    model.eval(); rows = split_obs_rows[split]; nodes = splits[split]
    with torch.no_grad():
        air_pred, mode_pred = model(x, edge_index, edge_type_t)
        loss = ds_kl(air_pred[nodes], y_air_t[rows]) + ds_kl(mode_pred[nodes], y_mode_t[rows])
        air_hat = torch.argmax(air_pred[nodes][:, singleton_score_indices(len(aircraft_classes))], dim=1)
        mode_hat = torch.argmax(mode_pred[nodes][:, singleton_score_indices(len(mode_classes))], dim=1)
        air_true = torch.tensor([aircraft_to_idx[observations[int(r.cpu())]["ground_truth_label"]["aircraft_variant"]] for r in rows], device=DEVICE)
        mode_true = torch.tensor([mode_to_idx[observations[int(r.cpu())]["ground_truth_label"]["mode"]] for r in rows], device=DEVICE)
        return {"loss": float(loss.cpu()), "aircraft_acc": float((air_hat==air_true).float().mean().cpu()), "mode_acc": float((mode_hat==mode_true).float().mean().cpu())}
best_val = math.inf; bad_epochs = 0; history = []; ckpt_path = ARTIFACT_DIR / "best_rgcn_ds_model.pt"
if SummaryWriter is None:
    raise ImportError("TensorBoard tracking requires the tensorboard package. Install dependencies with `pip install -r requirements.txt`.")
writer = SummaryWriter(log_dir=str(TENSORBOARD_LOG_DIR))
writer.add_text("run/config", json.dumps({"epochs": EPOCHS, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "hidden_features": HIDDEN_FEATURES, "dropout": DROPOUT, "train_fraction": TRAIN_FRACTION, "test_fraction": TEST_FRACTION, "val_fraction": VAL_FRACTION}, indent=2))
try:
    for epoch in tqdm(range(1, EPOCHS + 1), desc="Training r-GCN"):
        model.train(); optimizer.zero_grad(set_to_none=True)
        air_pred, mode_pred = model(x, edge_index, edge_type_t)
        tr_nodes, tr_rows = splits["train"], split_obs_rows["train"]
        loss = ds_kl(air_pred[tr_nodes], y_air_t[tr_rows]) + ds_kl(mode_pred[tr_nodes], y_mode_t[tr_rows])
        loss.backward(); optimizer.step()
        val = eval_split("val")
        train_loss = float(loss.detach().cpu())
        history.append({"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k,v in val.items()}})
        writer.add_scalar("loss/train", train_loss, epoch)
        writer.add_scalar("loss/validation", val["loss"], epoch)
        writer.add_scalar("accuracy_validation/aircraft_variant", val["aircraft_acc"], epoch)
        writer.add_scalar("accuracy_validation/radar_mode", val["mode_acc"], epoch)
        writer.add_scalar("optimizer/learning_rate", optimizer.param_groups[0]["lr"], epoch)
        if val["loss"] < best_val:
            best_val = val["loss"]; bad_epochs = 0
            torch.save({"model_state": model.state_dict(), "aircraft_classes": aircraft_classes, "mode_classes": mode_classes, "relations": relations, "frequency_gate_summary": frequency_gate_summary, "use_singleton_uncertainty_masses": use_singleton_uncertainty_masses, "config": {"epochs": EPOCHS, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "hidden_features": HIDDEN_FEATURES, "dropout": DROPOUT, "tensorboard_log_dir": str(TENSORBOARD_LOG_DIR)}}, ckpt_path)
        else:
            bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}"); break
finally:
    writer.flush(); writer.close()
history[-5:]


## Test-set evaluation and Dempster-Shafer belief-plausibility identifications

In [ ]:
checkpoint = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
print("Test metrics:", eval_split("test"))
def intervals_for(mass, classes):
    mass = validate_masses(mass)
    if use_singleton_uncertainty_masses:
        uncertainty = float(mass[-1])
        return sorted(({"hypothesis": cls, "belief": float(mass[i]), "plausibility": float(mass[i] + uncertainty)} for i, cls in enumerate(classes)), key=lambda r: r["belief"], reverse=True)
    return sorted((i.__dict__ for i in belief_plausibility(mass, classes)), key=lambda r: r["belief"], reverse=True)
model.eval()
with torch.no_grad():
    air_pred, mode_pred = model(x, edge_index, edge_type_t)
    air_np = air_pred.cpu().numpy(); mode_np = mode_pred.cpu().numpy()
results = []
for node_i in splits["test"].detach().cpu().numpy():
    row_i = obs_pos[int(node_i)]; obs = observations[row_i]
    air_intervals = intervals_for(air_np[node_i], aircraft_classes); mode_intervals = intervals_for(mode_np[node_i], mode_classes)
    results.append({"observation_id": obs["observation_id"], "truth_aircraft_variant": obs["ground_truth_label"]["aircraft_variant"], "truth_radar_mode": obs["ground_truth_label"]["mode"], "pred_aircraft_variant": air_intervals[0]["hypothesis"], "pred_aircraft_belief_plausibility": (air_intervals[0]["belief"], air_intervals[0]["plausibility"]), "pred_radar_mode": mode_intervals[0]["hypothesis"], "pred_mode_belief_plausibility": (mode_intervals[0]["belief"], mode_intervals[0]["plausibility"]), "top3_aircraft_intervals": air_intervals[:3], "top3_mode_intervals": mode_intervals[:3]})
(ARTIFACT_DIR / "test_identifications.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "training_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
results
